# SI10-2026 | Ponderada | Analise de Sensibilidade em Metricas de Interface Digital

Este notebook analisa quais metricas de interface tem maior impacto sobre a taxa de conversao em um aplicativo de compras. A entrega combina codigo, tabelas, graficos e respostas curtas, mantendo a separacao entre dado simulado, modelo ajustado, analise de sensibilidade e recomendacao de produto.

## Contexto

Uma equipe de produto quer decidir qual metrica de interface deve receber prioridade no proximo ciclo de melhoria. Os dados representam observacoes diarias de um aplicativo de compras. A metrica alvo e `taxa_conversao_pct`.

Variaveis de entrada:

- `taxa_abandono_carrinho_pct`: percentual de abandono no carrinho.
- `profundidade_scroll_pct`: profundidade media de scroll.
- `tempo_primeiro_clique_s`: tempo ate o primeiro clique em produto.

## Hipoteses de trabalho

- **H1:** maior abandono do carrinho deve reduzir a taxa de conversao.
- **H2:** maior profundidade de scroll pode aumentar a conversao quando representa exploracao produtiva da vitrine.
- **H3:** maior tempo ate o primeiro clique deve reduzir a conversao por indicar friccao inicial.
- **H4:** pequenas melhorias simultaneas em friccoes diferentes podem gerar ganho plausivel, mas ainda precisam ser validadas por teste controlado.

## Preparacao

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.precision", 3)

## Dados

A celula abaixo cria a base da atividade. A semente aleatoria fixa garante que os resultados sejam reprodutiveis.

In [2]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


### Definicao das variaveis

As variaveis sao definidas logo apos a criacao da base para manter os mesmos nomes na exploracao, no modelo, na sensibilidade e na simulacao.

In [3]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

resumo_base = df[features + [target]].describe().T
resumo_base

,count,mean,std,min,25%,50%,75%,max
taxa_abandono_carrinho_pct,180.0,47.546,6.939,30.944,42.697,47.466,51.916,71.311
profundidade_scroll_pct,180.0,62.449,12.166,31.200,54.099,62.511,69.722,95.000
tempo_primeiro_clique_s,180.0,6.971,2.244,2.000,5.477,7.072,8.466,12.715
taxa_conversao_pct,180.0,5.869,0.645,4.352,5.434,5.916,6.307,7.753


## Parte 1 | Exploracao

A exploracao inicial verifica sinal, magnitude e estabilidade das relacoes. A escolha das variaveis para sensibilidade nao deve sair apenas da intuicao; precisa aparecer nos dados.

In [4]:
colunas_numericas = features + [target]
correlacoes = df[colunas_numericas].corr()

correlacao_com_conversao = (
    correlacoes[target]
    .drop(target)
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .reset_index()
    .rename(columns={"index": "variavel", target: "correlacao_com_taxa_conversao"})
)

print("Matriz de correlacao")
display(correlacoes)

print("Ranking por correlacao absoluta com a conversao")
display(correlacao_com_conversao)

Matriz de correlacao


,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


Ranking por correlacao absoluta com a conversao


,variavel,correlacao_com_taxa_conversao
0,taxa_abandono_carrinho_pct,-0.643
1,profundidade_scroll_pct,0.485
2,tempo_primeiro_clique_s,-0.229


In [5]:
fig = px.imshow(
    correlacoes,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Mapa de correlacao entre metricas de interface e conversao",
)
fig.update_layout(width=850, height=600)
fig.show()

In [6]:
fig = px.bar(
    correlacao_com_conversao.sort_values("correlacao_com_taxa_conversao"),
    x="correlacao_com_taxa_conversao",
    y="variavel",
    orientation="h",
    color="correlacao_com_taxa_conversao",
    color_continuous_scale="RdBu",
    title="Relacao das variaveis com a taxa de conversao",
    labels={"correlacao_com_taxa_conversao": "Correlacao com conversao", "variavel": "Variavel"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.show()

In [25]:
fig = px.scatter(
    df,
    x="taxa_abandono_carrinho_pct",
    y=target,
    title="Abandono do carrinho vs. taxa de conversao",
    labels={"taxa_abandono_carrinho_pct": "Taxa de abandono do carrinho (%)", target: "Taxa de conversao (%)"},
)
fig.show()

fig = px.scatter(
    df,
    x="profundidade_scroll_pct",
    y=target,
    title="Profundidade de scroll vs. taxa de conversao",
    labels={"profundidade_scroll_pct": "Profundidade de scroll (%)", target: "Taxa de conversao (%)"},
)
fig.show()

In [7]:
sinais_hipoteses = pd.DataFrame({
    "hipotese": ["H1", "H2", "H3"],
    "variavel": features,
    "sinal_esperado": ["negativo", "positivo", "negativo"],
    "correlacao_observada": [
        correlacoes.loc["taxa_abandono_carrinho_pct", target],
        correlacoes.loc["profundidade_scroll_pct", target],
        correlacoes.loc["tempo_primeiro_clique_s", target],
    ],
})
sinais_hipoteses["sinal_observado"] = np.where(sinais_hipoteses["correlacao_observada"] >= 0, "positivo", "negativo")
sinais_hipoteses["hipotese_confirmada_no_sinal"] = sinais_hipoteses["sinal_esperado"] == sinais_hipoteses["sinal_observado"]
sinais_hipoteses

,hipotese,variavel,sinal_esperado,correlacao_observada,sinal_observado,hipotese_confirmada_no_sinal
0,H1,taxa_abandono_carrinho_pct,negativo,-0.643,negativo,True
1,H2,profundidade_scroll_pct,positivo,0.485,positivo,True
2,H3,tempo_primeiro_clique_s,negativo,-0.229,negativo,True


**Resposta - escolha das variaveis**

Escolhi `taxa_abandono_carrinho_pct` e `profundidade_scroll_pct` para a analise principal. Na exploracao, elas foram as duas variaveis com maior relacao com `taxa_conversao_pct`: abandono do carrinho com correlacao negativa de aproximadamente **-0,643** e profundidade de scroll com correlacao positiva de aproximadamente **0,485**.

A escolha tambem faz sentido para produto porque as duas variaveis representam pontos diferentes da experiencia. Abandono do carrinho esta ligado ao fim do funil; profundidade de scroll indica quanto o usuario percorre a vitrine antes de converter. O `tempo_primeiro_clique_s` tambem foi analisado, mas apareceu com relacao mais fraca neste conjunto de dados.

## Parte 2 | Modelo

Ajuste de um modelo linear para estimar a taxa de conversao a partir das metricas de interface. O modelo e usado para comparar cenarios locais, nao para afirmar causalidade.

In [8]:
X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])
coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

metricas_modelo = pd.DataFrame({"metrica": ["MAE", "RMSE"], "valor": [mae, rmse]})
coeficientes_modelo = pd.DataFrame({"termo": ["intercepto"] + features, "coeficiente": coeficientes})

display(metricas_modelo)
display(coeficientes_modelo)

,metrica,valor
0,MAE,0.276
1,RMSE,0.344


,termo,coeficiente
0,intercepto,7.883
1,taxa_abandono_carrinho_pct,-0.060
2,profundidade_scroll_pct,0.024
3,tempo_primeiro_clique_s,-0.094


In [9]:
coef_padronizados = pd.DataFrame({
    "variavel": features,
    "coeficiente_original": coeficientes[1:],
    "desvio_padrao_variavel": df[features].std().values,
    "coeficiente_padronizado": coeficientes[1:] * df[features].std().values / df[target].std(),
})
coef_padronizados["impacto_absoluto"] = coef_padronizados["coeficiente_padronizado"].abs()
coef_padronizados = coef_padronizados.sort_values("impacto_absoluto", ascending=False)

display(coef_padronizados)

fig = px.bar(
    coef_padronizados.sort_values("impacto_absoluto"),
    x="coeficiente_padronizado",
    y="variavel",
    orientation="h",
    color="coeficiente_padronizado",
    color_continuous_scale="RdBu",
    title="Coeficientes padronizados do modelo",
    labels={"coeficiente_padronizado": "Coeficiente padronizado", "variavel": "Variavel"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.show()

,variavel,coeficiente_original,desvio_padrao_variavel,coeficiente_padronizado,impacto_absoluto
0,taxa_abandono_carrinho_pct,-0.060,6.939,-0.650,0.650
1,profundidade_scroll_pct,0.024,12.166,0.457,0.457
2,tempo_primeiro_clique_s,-0.094,2.244,-0.328,0.328


In [10]:
df_modelo = df.copy()
df_modelo["conversao_prevista"] = pred
df_modelo["residuo"] = erro

resumo_residuos = df_modelo["residuo"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
display(resumo_residuos.to_frame("residuo"))

fig = px.scatter(
    df_modelo,
    x="conversao_prevista",
    y=target,
    title="Conversao observada vs. conversao prevista",
    labels={"conversao_prevista": "Conversao prevista (%)", target: "Conversao observada (%)"},
)
min_ref = min(df_modelo["conversao_prevista"].min(), df_modelo[target].min())
max_ref = max(df_modelo["conversao_prevista"].max(), df_modelo[target].max())
fig.add_trace(go.Scatter(x=[min_ref, max_ref], y=[min_ref, max_ref], mode="lines", name="previsao perfeita"))
fig.show()

fig = px.histogram(df_modelo, x="residuo", nbins=30, title="Distribuicao dos residuos do modelo", labels={"residuo": "Erro observado - previsto (p.p.)"})
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.show()

,residuo
count,1.800e+02
mean,-1.638e-15
std,3.446e-01
min,-8.379e-01
10%,-4.364e-01
25%,-2.193e-01
50%,-6.908e-03
75%,2.411e-01
90%,4.244e-01
max,9.511e-01


**Resposta - erro do modelo**

O modelo apresentou **MAE de aproximadamente 0,276 p.p.** e **RMSE de aproximadamente 0,344 p.p.**. Como a taxa media de conversao da base fica perto de **5,87%**, o erro e baixo o suficiente para comparar cenarios locais de sensibilidade.

O RMSE ficou um pouco acima do MAE, o que e esperado porque ele penaliza mais erros grandes. Como os dois valores ainda estao proximos, nao ha sinal forte de que poucos erros extremos estejam dominando a avaliacao do modelo.

## Parte 3 | Analise de Sensibilidade

A sensibilidade mede quanto a saida muda quando uma entrada muda. Aqui, a variacao principal e de 10%, como solicitado na atividade.

In [11]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)

linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)
linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [12]:
variaveis_escolhidas = ["taxa_abandono_carrinho_pct", "profundidade_scroll_pct"]

variacao_entrada = 0.10
resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado
    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada
    resultados.append({
        "variavel": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saida_original": saida_base,
        "saida_nova": saida_nova,
        "diferenca_pp": saida_nova - saida_base,
        "variacao_saida_pct": variacao_saida * 100,
        "indice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

,variavel,valor_original,valor_alterado,saida_original,saida_nova,diferenca_pp,variacao_saida_pct,indice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-0.287,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,0.151,2.578,0.258


**Resposta - comparacao dos indices**

A variavel com maior impacto local foi `taxa_abandono_carrinho_pct`. Um aumento de **10%** nessa entrada reduziu a conversao prevista em cerca de **4,89%**, com indice de sensibilidade de aproximadamente **-0,489**. O sinal negativo indica que aumento no abandono reduz a conversao.

A segunda variavel escolhida, `profundidade_scroll_pct`, teve indice de aproximadamente **0,258**: um aumento de **10%** elevou a conversao prevista em cerca de **2,58%**. Para decisao de produto, o abandono do carrinho deve receber prioridade porque tem maior impacto absoluto no modelo.

### Ir alem 1 | Ranking completo de sensibilidade

A atividade pede duas variaveis, mas o ranking abaixo calcula a sensibilidade das tres entradas. Isso evita escolher variaveis por conveniencia e mostra se a decisao se mantem quando todas sao comparadas pelo mesmo criterio.

In [13]:
resultados_completos = []
for variavel in features:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * 1.10
    linha_cenario[variavel] = valor_alterado
    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / 0.10
    resultados_completos.append({
        "variavel": variavel,
        "saida_base": saida_base,
        "saida_mais_10_pct": saida_nova,
        "diferenca_pp": saida_nova - saida_base,
        "variacao_saida_pct": variacao_saida * 100,
        "indice_sensibilidade": indice_sensibilidade,
        "impacto_absoluto": abs(indice_sensibilidade),
    })
ranking_sensibilidade = pd.DataFrame(resultados_completos).sort_values("impacto_absoluto", ascending=False).reset_index(drop=True)
ranking_sensibilidade

,variavel,saida_base,saida_mais_10_pct,diferenca_pp,variacao_saida_pct,indice_sensibilidade,impacto_absoluto
0,taxa_abandono_carrinho_pct,5.869,5.582,-0.287,-4.891,-0.489,0.489
1,profundidade_scroll_pct,5.869,6.020,0.151,2.578,0.258,0.258
2,tempo_primeiro_clique_s,5.869,5.803,-0.066,-1.118,-0.112,0.112


In [14]:
fig = px.bar(
    ranking_sensibilidade.sort_values("impacto_absoluto"),
    x="indice_sensibilidade",
    y="variavel",
    orientation="h",
    color="indice_sensibilidade",
    color_continuous_scale="RdBu",
    title="Ranking de sensibilidade das variaveis de interface",
    labels={"indice_sensibilidade": "Indice de sensibilidade", "variavel": "Variavel"},
)
fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="black")
fig.show()

O ranking confirma que `taxa_abandono_carrinho_pct` e a variavel mais sensivel no modelo. `profundidade_scroll_pct` aparece em segundo lugar e `tempo_primeiro_clique_s` tem impacto menor neste conjunto, embora continue sendo uma metrica util de friccao inicial.

### Ir alem 2 | Direcao da mudanca (+10% e -10%)

A atividade pede aumento de 10%, mas varias decisoes de interface buscam reduzir uma friccao. Por isso, a tabela abaixo testa tambem a reducao de 10%.

In [15]:
cenarios_direcao = []
for variavel in features:
    for variacao in [0.10, -0.10]:
        linha_cenario = linha_base.copy()
        valor_original = linha_base[variavel]
        valor_alterado = valor_original * (1 + variacao)
        linha_cenario[variavel] = valor_alterado
        saida_nova = prever_linha(linha_cenario)
        variacao_saida = (saida_nova - saida_base) / saida_base
        indice_sensibilidade = variacao_saida / variacao
        cenarios_direcao.append({
            "variavel": variavel,
            "variacao_entrada_pct": variacao * 100,
            "valor_alterado": valor_alterado,
            "taxa_conversao_prevista": saida_nova,
            "diferenca_pp": saida_nova - saida_base,
            "variacao_saida_pct": variacao_saida * 100,
            "indice_sensibilidade": indice_sensibilidade,
        })
tabela_direcao = pd.DataFrame(cenarios_direcao)
tabela_direcao

,variavel,variacao_entrada_pct,valor_alterado,taxa_conversao_prevista,diferenca_pp,variacao_saida_pct,indice_sensibilidade
0,taxa_abandono_carrinho_pct,10.0,52.300,5.582,-0.287,-4.891,-0.489
1,taxa_abandono_carrinho_pct,-10.0,42.791,6.156,0.287,4.891,-0.489
2,profundidade_scroll_pct,10.0,68.694,6.020,0.151,2.578,0.258
3,profundidade_scroll_pct,-10.0,56.204,5.717,-0.151,-2.578,0.258
4,tempo_primeiro_clique_s,10.0,7.668,5.803,-0.066,-1.118,-0.112
5,tempo_primeiro_clique_s,-10.0,6.274,5.934,0.066,1.118,-0.112


In [16]:
fig = px.bar(
    tabela_direcao,
    x="diferenca_pp",
    y="variavel",
    color="variacao_entrada_pct",
    barmode="group",
    orientation="h",
    title="Impacto previsto para aumento e reducao de 10%",
    labels={"diferenca_pp": "Diferenca na conversao prevista (p.p.)", "variavel": "Variavel", "variacao_entrada_pct": "Variacao da entrada (%)"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.show()

A direcao dos efeitos fica coerente com a interpretacao de produto. Reduzir abandono do carrinho aumenta a conversao prevista; aumentar profundidade de scroll tambem aumenta a conversao no modelo; reduzir tempo ate o primeiro clique gera ganho menor, mas na direcao esperada.

### Ir alem 3 | Quanto precisa mudar para gerar ganho visivel?

A sensibilidade mostra impacto relativo. A tabela abaixo traduz isso para uma pergunta de produto: quanto cada variavel teria que mudar para aumentar a conversao prevista em 0,20 p.p. ou 0,50 p.p.?

In [17]:
metas_pp = [0.20, 0.50]
linhas_meta = []
for variavel, coef in zip(features, coeficientes[1:]):
    for meta in metas_pp:
        mudanca_necessaria = meta / coef
        valor_base = linha_base[variavel]
        valor_alvo = valor_base + mudanca_necessaria
        linhas_meta.append({
            "variavel": variavel,
            "meta_ganho_pp": meta,
            "valor_base": valor_base,
            "mudanca_necessaria": mudanca_necessaria,
            "valor_alvo": valor_alvo,
            "mudanca_relativa_pct": mudanca_necessaria / valor_base * 100,
            "leitura": "reduzir" if mudanca_necessaria < 0 else "aumentar",
        })
tabela_metas = pd.DataFrame(linhas_meta)
tabela_metas

,variavel,meta_ganho_pp,valor_base,mudanca_necessaria,valor_alvo,mudanca_relativa_pct,leitura
0,taxa_abandono_carrinho_pct,0.2,47.546,-3.313,44.233,-6.968,reduzir
1,taxa_abandono_carrinho_pct,0.5,47.546,-8.282,39.264,-17.419,reduzir
2,profundidade_scroll_pct,0.2,62.449,8.256,70.705,13.220,aumentar
3,profundidade_scroll_pct,0.5,62.449,20.639,83.088,33.050,aumentar
4,tempo_primeiro_clique_s,0.2,6.971,-2.124,4.847,-30.473,reduzir
5,tempo_primeiro_clique_s,0.5,6.971,-5.311,1.660,-76.183,reduzir


Essa traducao ajuda a priorizar. Para ganhar **0,20 p.p.**, o modelo sugere que seria necessario reduzir abandono do carrinho em cerca de **3,31 p.p.**, aumentar scroll em cerca de **8,26 p.p.** ou reduzir tempo ate o primeiro clique em cerca de **2,12 s**. A primeira alternativa parece mais diretamente acionavel no fluxo de produto.

### Ir alem 4 | Cenarios combinados conservadores

Produto normalmente altera mais de uma friccao ao mesmo tempo. Os cenarios abaixo testam combinacoes pequenas, sem tratar o resultado como previsao fechada.

In [18]:
cenarios_produto = []

def avaliar_cenario(nome, ajustes):
    linha = linha_base.copy()
    for variavel, multiplicador in ajustes.items():
        linha[variavel] *= multiplicador
    saida = prever_linha(linha)
    cenarios_produto.append({
        "cenario": nome,
        "conversao_prevista": saida,
        "diferenca_pp": saida - saida_base,
        "variacao_pct_vs_base": (saida - saida_base) / saida_base * 100,
    })

avaliar_cenario("base", {})
avaliar_cenario("reduzir abandono 5pct", {"taxa_abandono_carrinho_pct": 0.95})
avaliar_cenario("reduzir primeiro clique 5pct", {"tempo_primeiro_clique_s": 0.95})
avaliar_cenario("aumentar scroll 5pct", {"profundidade_scroll_pct": 1.05})
avaliar_cenario("combinado conservador", {"taxa_abandono_carrinho_pct": 0.95, "tempo_primeiro_clique_s": 0.95})
avaliar_cenario("combinado vitrine e carrinho", {"taxa_abandono_carrinho_pct": 0.95, "profundidade_scroll_pct": 1.03, "tempo_primeiro_clique_s": 0.95})

tabela_cenarios_produto = pd.DataFrame(cenarios_produto)
tabela_cenarios_produto

,cenario,conversao_prevista,diferenca_pp,variacao_pct_vs_base
0,base,5.869,0.000,0.000
1,reduzir abandono 5pct,6.012,0.144,2.446
2,reduzir primeiro clique 5pct,5.902,0.033,0.559
3,aumentar scroll 5pct,5.944,0.076,1.289
4,combinado conservador,6.045,0.176,3.005
5,combinado vitrine e carrinho,6.090,0.222,3.778


In [19]:
fig = px.bar(
    tabela_cenarios_produto,
    x="cenario",
    y="diferenca_pp",
    title="Ganho previsto por cenario de produto",
    labels={"cenario": "Cenario", "diferenca_pp": "Diferenca vs. base (p.p.)"},
)
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(xaxis_tickangle=-25)
fig.show()

No cenario combinado conservador, a conversao prevista sobe de aproximadamente **5,87%** para **6,05%**, um ganho de cerca de **0,18 p.p.**. A leitura correta e de plausibilidade: pequenas reducoes em friccoes diferentes podem se somar, mas a confirmacao ainda depende de teste real de interface.

## Parte 4 | Decisao

A recomendacao deve sair da sensibilidade e ser acompanhada de metricas de validacao. A tabela abaixo transforma resultado analitico em plano de produto.

In [20]:
matriz_decisao = pd.DataFrame([
    {"prioridade": 1, "acao": "reduzir abandono no carrinho", "variavel_alvo": "taxa_abandono_carrinho_pct", "evidencia": "maior sensibilidade absoluta", "metrica_de_validacao": "conversao e abandono do carrinho", "risco": "efeito pode depender de preco, frete ou estoque"},
    {"prioridade": 2, "acao": "melhorar organizacao da vitrine", "variavel_alvo": "profundidade_scroll_pct", "evidencia": "segunda maior sensibilidade e sinal positivo", "metrica_de_validacao": "scroll, clique em produto e conversao", "risco": "scroll alto pode ser exploracao ou dificuldade"},
    {"prioridade": 3, "acao": "reduzir tempo ate primeiro clique", "variavel_alvo": "tempo_primeiro_clique_s", "evidencia": "sinal negativo, mas menor impacto no modelo", "metrica_de_validacao": "tempo ate clique e taxa de clique em produto", "risco": "ganho isolado pode ser pequeno"},
])
matriz_decisao

,prioridade,acao,variavel_alvo,evidencia,metrica_de_validacao,risco
0,1,reduzir abandono no carrinho,taxa_abandono_carrinho_pct,maior sensibilidade absoluta,conversao e abandono do carrinho,"efeito pode depender de preco, frete ou estoque"
1,2,melhorar organizacao da vitrine,profundidade_scroll_pct,segunda maior sensibilidade e sinal positivo,"scroll, clique em produto e conversao",scroll alto pode ser exploracao ou dificuldade
2,3,reduzir tempo ate primeiro clique,tempo_primeiro_clique_s,"sinal negativo, mas menor impacto no modelo",tempo ate clique e taxa de clique em produto,ganho isolado pode ser pequeno


**Resposta - recomendacao e limitacao**

A recomendacao principal e priorizar uma melhoria no fluxo de carrinho e checkout, reduzindo pontos de abandono antes da finalizacao. A tabela de sensibilidade indica que uma variacao de **10%** no abandono do carrinho muda a conversao prevista em cerca de **4,89%**, maior impacto entre as variaveis avaliadas.

A limitacao e que o modelo estima uma relacao local em uma base simulada, nao um efeito causal observado em usuarios reais. Por isso, a recomendacao deve ser tratada como hipotese para teste: a validacao real exigiria medir abandono, conversao e tempo de conclusao antes e depois da mudanca na interface.

## Ao Alem dos Alens | Monte Carlo

A simulacao de Monte Carlo estima como a taxa de conversao pode variar sob incerteza nas variaveis de entrada. O resultado deve ser lido como distribuicao, nao como valor unico garantido.

In [21]:
n_simulacoes = 5000
amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(linha_base["profundidade_scroll_pct"], 8, n_simulacoes).clip(25, 95),
    "tempo_primeiro_clique_s": rng.normal(linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes).clip(2, 15),
})
amostras_design = np.column_stack([np.ones(len(amostras)), amostras[features].to_numpy()])
previsoes = amostras_design @ coeficientes
resumo_monte_carlo = pd.Series(previsoes).describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
prob_abaixo_base = (previsoes < saida_base).mean()
print(f"Probabilidade simulada de ficar abaixo da linha base: {prob_abaixo_base:.1%}")
resumo_monte_carlo

Probabilidade simulada de ficar abaixo da linha base: 49.1%


,0
count,5000.000
mean,5.872
std,0.389
min,4.112
5%,5.237
10%,5.374
25%,5.614
50%,5.878
75%,6.131
90%,6.367


In [22]:
df_previsoes = pd.DataFrame({"taxa_conversao_pct_prevista": previsoes})
fig = px.histogram(
    df_previsoes,
    x="taxa_conversao_pct_prevista",
    nbins=35,
    title="Distribuicao simulada da taxa de conversao prevista",
    labels={"taxa_conversao_pct_prevista": "Taxa de conversao prevista (%)"},
)
fig.add_vline(x=saida_base, line_dash="dash", line_color="black", annotation_text="linha base")
fig.add_vline(x=np.percentile(previsoes, 10), line_dash="dot", line_color="red", annotation_text="P10")
fig.add_vline(x=np.percentile(previsoes, 90), line_dash="dot", line_color="green", annotation_text="P90")
fig.show()

### Comparacao Monte Carlo: baseline vs. intervencao conservadora

Para ligar a simulacao a decisao de produto, o bloco abaixo compara a distribuicao da linha base com a distribuicao de um cenario conservador: reduzir abandono do carrinho em 5% e reduzir tempo ate primeiro clique em 5%.

In [23]:
amostras_intervencao = amostras.copy()
amostras_intervencao["taxa_abandono_carrinho_pct"] *= 0.95
amostras_intervencao["tempo_primeiro_clique_s"] *= 0.95
previsoes_intervencao = np.column_stack([np.ones(len(amostras_intervencao)), amostras_intervencao[features].to_numpy()]) @ coeficientes
delta_intervencao = previsoes_intervencao - previsoes
comparacao_mc = pd.DataFrame({
    "metrica": ["media_base", "media_intervencao", "ganho_medio_pp", "p10_ganho_pp", "p50_ganho_pp", "p90_ganho_pp", "prob_ganho_positivo"],
    "valor": [previsoes.mean(), previsoes_intervencao.mean(), delta_intervencao.mean(), np.percentile(delta_intervencao, 10), np.percentile(delta_intervencao, 50), np.percentile(delta_intervencao, 90), (delta_intervencao > 0).mean()],
})
comparacao_mc

,metrica,valor
0,media_base,5.872
1,media_intervencao,6.048
2,ganho_medio_pp,0.176
3,p10_ganho_pp,0.155
4,p50_ganho_pp,0.176
5,p90_ganho_pp,0.197
6,prob_ganho_positivo,1.000


In [24]:
df_comparacao_mc = pd.DataFrame({
    "taxa_conversao_pct_prevista": np.concatenate([previsoes, previsoes_intervencao]),
    "cenario": ["base"] * len(previsoes) + ["intervencao_conservadora"] * len(previsoes_intervencao),
})
fig = px.histogram(
    df_comparacao_mc,
    x="taxa_conversao_pct_prevista",
    color="cenario",
    barmode="overlay",
    opacity=0.55,
    nbins=35,
    title="Monte Carlo: baseline vs. intervencao conservadora",
    labels={"taxa_conversao_pct_prevista": "Taxa de conversao prevista (%)"},
)
fig.show()

**Resposta - risco da recomendacao**

A simulacao de Monte Carlo mostra que a conversao prevista fica concentrada perto de **5,88%**, mas com variacao relevante: o intervalo entre P10 e P90 ficou aproximadamente entre **5,37%** e **6,37%**. Isso indica que a recomendacao nao deve ser lida como ganho fixo.

Quando a intervencao conservadora e simulada, o ganho medio fica perto de **0,18 p.p.** e a direcao permanece positiva. Ainda assim, isso continua sendo uma estimativa do modelo. O proximo passo correto seria um teste controlado medindo conversao, abandono do carrinho e tempo ate primeiro clique.

## Fechamento

A prioridade mais defensavel e reduzir abandono no carrinho, porque essa variavel teve maior relacao negativa com conversao, maior coeficiente padronizado em modulo e maior indice de sensibilidade. A profundidade de scroll aparece como segunda frente, mas precisa ser interpretada com cuidado: scroll maior pode representar exploracao produtiva ou dificuldade de encontrar produto.

A entrega nao trata a simulacao como prova causal. Ela organiza uma hipotese de produto, mostra o impacto previsto sob diferentes cenarios e define o que deve ser medido em um teste real.

## Politica de uso de IA

O uso de IA e permitido para apoio tecnico, revisao de texto e estudo dos conceitos. As escolhas de variaveis, os calculos, a comparacao dos indices e a recomendacao devem refletir a analise dos resultados deste notebook.

## Instrucoes de entrega

A entrega deve ser feita no GitHub ou no proprio Google Colab. Links sem permissao de acesso podem ter desconto de nota.